# Hyperparameter Tuning — VAE Alone

**New notebook.** `train_vae.ipynb` only ever ran a *coordinate-wise* search:
fix `latent_dim=16`, search `beta_max` over 4 coarse points; then fix the
chosen `beta_max`, search `latent_dim` over {8, 16, 32}. This never tests
whether a *combination* away from that path (e.g. `latent_dim=24` at
`beta_max=0.007`) might do better — the two hyperparameters can interact
(a larger latent space accumulates more total KL for the same per-dimension
behavior, so the beta that avoids collapse at one latent_dim may not be
optimal at another).

**This notebook runs a joint grid** over both hyperparameters together, picks
the best config by validation loss alone (leak-free — `cc1_val` has zero
anomalies), fully trains that winner, and evaluates it with the exact same
protocol as `vae_eval.ipynb` — so the result is directly comparable to the
currently-deployed model's numbers.

**Scope, stated up front**: `hidden1=64`, `hidden2=32` (architecture width),
learning rate, and batch size are held fixed at their existing values — only
`latent_dim` and `beta_max` are tuned here, since those are the two
hyperparameters the project's own prior ablations already identified as most
consequential. Widening the search to layer sizes/learning rate is a
reasonable further step, not done here due to the added compute cost.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, confusion_matrix, precision_recall_curve)

torch.manual_seed(42)
np.random.seed(42)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')

DEVICE = torch.device('cpu')
ALL_SETS = ['cc1_test', 'drift_cc2']

HIDDEN1, HIDDEN2 = 64, 32
WARMUP_EPOCHS = 10
SCREEN_EPOCHS, SCREEN_PATIENCE = 40, 8
FULL_MAX_EPOCHS, FULL_PATIENCE = 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0

LATENT_GRID = [16, 20, 24, 32, 40]
BETA_GRID   = [0.003, 0.005, 0.007, 0.01, 0.015]

print(f'Grid: {len(LATENT_GRID)} x {len(BETA_GRID)} = {len(LATENT_GRID)*len(BETA_GRID)} combinations')
print(f'LATENT_GRID={LATENT_GRID}')
print(f'BETA_GRID={BETA_GRID}')

Grid: 5 x 5 = 25 combinations
LATENT_GRID=[16, 20, 24, 32, 40]
BETA_GRID=[0.003, 0.005, 0.007, 0.01, 0.015]


## Step 1 — Load data (same sanity-guard discipline as `train_vae.ipynb`)

In [2]:
raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]

for name, X in raw.items():
    assert not np.isnan(X).any() and not np.isinf(X).any(), f'{name}: NaN/Inf found'
assert (labels['cc1_train'] == 0).all()

X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}  X_train={X_train_t.shape}  X_val={X_val_t.shape}')
print('Sanity checks passed.')

INPUT_DIM=26  X_train=torch.Size([154198, 26])  X_val=torch.Size([21573, 26])
Sanity checks passed.


## Step 2 — VAE class + training loop (identical to `train_vae.ipynb`, corrected KL formula)

In [3]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

def train_vae(latent_dim, beta_max, max_epochs, patience, seed=42):
    torch.manual_seed(seed)
    model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, latent_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0
    final_kl, final_recon = None, None

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * beta_max
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, rloss, kl = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            vloss, vrecon, vkl = vae_loss(recon, X_val_t, mu, logvar, beta)
        final_kl, final_recon = vkl.item(), vrecon.item()
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, final_kl, final_recon, epoch + 1

print('Training function defined.')

Training function defined.


## Step 3 — Joint grid screen (short budget: 40 epochs, patience 8 — same screening budget `train_vae.ipynb` used for its own ablations)

Every combination is trained fresh at the SAME seed (42), matching the
original ablation's convention, so any difference reflects the hyperparameter
choice, not seed variance. Collapsed runs (final KL < 0.05) are excluded from
selection, same collapse criterion as `train_vae.ipynb`.

In [4]:
grid_results = {}
t0_all = time.time()
for latent_dim in LATENT_GRID:
    for beta_max in BETA_GRID:
        t0 = time.time()
        _, best_val, final_kl, final_recon, n_epochs = train_vae(latent_dim, beta_max, SCREEN_EPOCHS, SCREEN_PATIENCE)
        elapsed = time.time() - t0
        collapsed = final_kl < 0.05
        grid_results[(latent_dim, beta_max)] = {
            'best_val': best_val, 'final_kl': final_kl, 'final_recon': final_recon,
            'collapsed': collapsed, 'epochs': n_epochs, 'time': elapsed,
        }
        flag = 'COLLAPSED' if collapsed else 'ok'
        print(f'  latent={latent_dim:3d}  beta={beta_max:.3f}  val_loss={best_val:.4f}  '
              f'final_kl={final_kl:.4f}  {flag}  ({elapsed:.0f}s)')
print(f'\nTotal screening time: {(time.time()-t0_all)/60:.1f} min')

  latent= 16  beta=0.003  val_loss=0.1247  final_kl=30.5535  ok  (79s)
  latent= 16  beta=0.005  val_loss=0.1751  final_kl=25.4560  ok  (75s)
  latent= 16  beta=0.007  val_loss=0.2213  final_kl=22.7098  ok  (81s)
  latent= 16  beta=0.010  val_loss=0.2758  final_kl=19.1282  ok  (80s)
  latent= 16  beta=0.015  val_loss=0.3553  final_kl=15.8497  ok  (81s)
  latent= 20  beta=0.003  val_loss=0.1243  final_kl=30.5093  ok  (83s)
  latent= 20  beta=0.005  val_loss=0.1780  final_kl=25.6170  ok  (86s)
  latent= 20  beta=0.007  val_loss=0.2192  final_kl=22.3479  ok  (90s)
  latent= 20  beta=0.010  val_loss=0.2766  final_kl=19.0925  ok  (91s)
  latent= 20  beta=0.015  val_loss=0.3475  final_kl=15.3926  ok  (86s)
  latent= 24  beta=0.003  val_loss=0.1313  final_kl=32.1359  ok  (83s)
  latent= 24  beta=0.005  val_loss=0.1703  final_kl=24.6081  ok  (82s)
  latent= 24  beta=0.007  val_loss=0.2169  final_kl=22.1327  ok  (80s)
  latent= 24  beta=0.010  val_loss=0.2721  final_kl=18.8180  ok  (81s)
  late

## Step 4 — Pick the best combination

In [5]:
candidates = {k: v for k, v in grid_results.items() if not v['collapsed']}
assert candidates, 'Every combination collapsed — grid needs adjustment.'

BEST_LATENT, BEST_BETA = min(candidates, key=lambda k: candidates[k]['best_val'])
print(f'Best combination: latent_dim={BEST_LATENT}, beta_max={BEST_BETA}')
print(f'  val_loss={candidates[(BEST_LATENT, BEST_BETA)]["best_val"]:.4f}')

# Compare to the ORIGINAL sequential-search result (latent_dim=32, beta_max=0.01) for reference
orig_key = (32, 0.01)
if orig_key in grid_results:
    print(f'\nFor reference, original config (latent=32, beta=0.01) at this SAME 40-epoch screening budget: '
          f'val_loss={grid_results[orig_key]["best_val"]:.4f}  collapsed={grid_results[orig_key]["collapsed"]}')

print('\nTop 5 combinations by val_loss:')
ranked = sorted(candidates.items(), key=lambda kv: kv[1]['best_val'])[:5]
for (ld, bm), r in ranked:
    print(f'  latent={ld:3d}  beta={bm:.3f}  val_loss={r["best_val"]:.4f}  final_kl={r["final_kl"]:.4f}')

Best combination: latent_dim=40, beta_max=0.003
  val_loss=0.1240

For reference, original config (latent=32, beta=0.01) at this SAME 40-epoch screening budget: val_loss=0.2652  collapsed=False

Top 5 combinations by val_loss:
  latent= 40  beta=0.003  val_loss=0.1240  final_kl=30.4141
  latent= 20  beta=0.003  val_loss=0.1243  final_kl=30.5093
  latent= 16  beta=0.003  val_loss=0.1247  final_kl=30.5535
  latent= 32  beta=0.003  val_loss=0.1271  final_kl=30.6719
  latent= 24  beta=0.003  val_loss=0.1313  final_kl=32.1359


## Step 5 — Full training of the winning configuration (300 epochs, patience 20 — matching `train_vae.ipynb`'s full-run budget)

In [6]:
print(f'Full training: latent_dim={BEST_LATENT}, beta_max={BEST_BETA} ...')
t0 = time.time()
tuned_model, tuned_val_loss, tuned_final_kl, tuned_final_recon, tuned_epochs = train_vae(
    BEST_LATENT, BEST_BETA, FULL_MAX_EPOCHS, FULL_PATIENCE)
elapsed = time.time() - t0
print(f'Done in {elapsed/60:.1f} min.  epochs={tuned_epochs}  val_loss={tuned_val_loss:.4f}  final_kl={tuned_final_kl:.4f}')
print(f'({"WARNING: looks collapsed" if tuned_final_kl < 0.05 else "non-trivial, ok"})')

Full training: latent_dim=40, beta_max=0.003 ...
Done in 11.4 min.  epochs=298  val_loss=0.1046  final_kl=25.4824
(non-trivial, ok)


## Step 6 — Full evaluation, identical protocol to `vae_eval.ipynb` — direct, honest comparison to the deployed model

In [7]:
with torch.no_grad():
    mse_train = tuned_model.anomaly_score(X_train_t).numpy()
    mse_val = tuned_model.anomaly_score(X_val_t).numpy()
mu_train, sigma_train = float(mse_train.mean()), float(mse_train.std())
val_p99 = float(np.percentile(mse_val, 99))
print(f'Tuned model: mu_train={mu_train:.5f}  sigma_train={sigma_train:.5f}  val_p99 threshold={val_p99:.5f}')

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

tuned_results = {}
mse_by_set = {}
for name in ALL_SETS:
    X_t = torch.from_numpy(data[name])
    with torch.no_grad():
        mse = tuned_model.anomaly_score(X_t).numpy()
    mse_by_set[name] = mse
    tuned_results[name] = evaluate(mse, labels[name], val_p99)

# The currently-deployed model's numbers, for direct comparison (from vae_cc1_eval.pkl, unchanged)
deployed_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))

print(f'\n{"set":12s} {"model":10s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"OracleF1":>9s}')
for name in ALL_SETS:
    t = tuned_results[name]
    d_auc = deployed_eval['auc'][name]
    d_pr = deployed_eval['precision_recall'][name]['val_p99']
    d_oracle = deployed_eval['oracle_ceiling'][name]['f1']
    print(f'{name:12s} {"tuned":10s} {t["auc_pr"]:8.4f} {t["auc_roc"]:9.4f} {t["f1"]:7.3f} {t["precision"]:10.3f} {t["recall"]:8.3f} {t["oracle_f1"]:9.4f}')
    print(f'{name:12s} {"deployed":10s} {d_auc["auc_pr"]:8.4f} {d_auc["auc_roc"]:9.4f} {d_pr["f1"]:7.3f} {d_pr["precision"]:10.3f} {d_pr["recall"]:8.3f} {d_oracle:9.4f}')
    print(f'  -> PR-AUC change: {t["auc_pr"]-d_auc["auc_pr"]:+.4f}   F1 change: {t["f1"]-d_pr["f1"]:+.3f}   Oracle-F1 change: {t["oracle_f1"]-d_oracle:+.4f}\n')

Tuned model: mu_train=0.00828  sigma_train=0.02029  val_p99 threshold=0.03935

set          model        PR-AUC   ROC-AUC      F1  Precision   Recall  OracleF1
cc1_test     tuned        0.5746    0.8628   0.574      0.577    0.570    0.6794
cc1_test     deployed     0.6014    0.8763   0.618      0.630    0.605    0.6857
  -> PR-AUC change: -0.0268   F1 change: -0.044   Oracle-F1 change: -0.0063

drift_cc2    tuned        0.3464    0.9009   0.244      0.145    0.767    0.4694
drift_cc2    deployed     0.4089    0.8812   0.276      0.168    0.772    0.5147
  -> PR-AUC change: -0.0625   F1 change: -0.031   Oracle-F1 change: -0.0453



## Step 7 — Per-fault-type recall, tuned vs. deployed

In [8]:
ft = {name: np.load(os.path.join(DATA_DIR, f'ft_{name}.npy'), allow_pickle=True) for name in ALL_SETS}
deployed_fault = deployed_eval['per_fault_recall']

print(f'{"set":12s} {"fault_type":14s} {"n":>5s} {"deployed recall":>16s} {"tuned recall":>13s}')
for name in ALL_SETS:
    pred = (mse_by_set[name] > val_p99).astype(int)
    types_present = sorted({v for v in ft[name] if isinstance(v, str)})
    for ftype in types_present:
        mask = ft[name] == ftype
        n = int(mask.sum())
        rec_tuned = pred[mask].mean() if n > 0 else float('nan')
        rec_deployed = deployed_fault[name][ftype]['recall']
        print(f'{name:12s} {ftype:14s} {n:5d} {rec_deployed:16.3f} {rec_tuned:13.3f}')
    print()

set          fault_type         n  deployed recall  tuned recall
cc1_test     cpu               88            0.580         0.568
cc1_test     memory            76            0.763         0.671
cc1_test     pod-failure       92            0.500         0.489

drift_cc2    cpu              207            0.908         0.855
drift_cc2    memory           414            0.713         0.725
drift_cc2    pod-failure       99            0.737         0.758



## Step 8 — Save results (does NOT overwrite the currently-deployed model/threshold)

In [9]:
save_results = {
    'grid_results': {f'{ld}_{bm}': v for (ld, bm), v in grid_results.items()},
    'best_config': {'latent_dim': BEST_LATENT, 'beta_max': BEST_BETA},
    'tuned_model_meta': {
        'input_dim': INPUT_DIM, 'hidden1': HIDDEN1, 'hidden2': HIDDEN2, 'latent_dim': BEST_LATENT,
        'beta_max': BEST_BETA, 'clip': CLIP, 'mu_train': mu_train, 'sigma_train': sigma_train,
        'val_p99': val_p99, 'epochs_trained': tuned_epochs,
    },
    'tuned_results': tuned_results,
    'deployed_comparison': {
        name: {'auc': deployed_eval['auc'][name], 'precision_recall': deployed_eval['precision_recall'][name]['val_p99'],
               'oracle_f1': deployed_eval['oracle_ceiling'][name]['f1']}
        for name in ALL_SETS
    },
}
out_path = os.path.join(OUT_DIR, 'hyperparameter_tuning_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

model_path = os.path.join(OUT_DIR, 'vae_cc1_tuned.pt')
torch.save(tuned_model.state_dict(), model_path)
print(f'Tuned model weights saved -> {model_path}  (NOT deployed — models/vae_cc1.pt is unchanged)')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\hyperparameter_tuning_results.pkl
Tuned model weights saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\vae_cc1_tuned.pt  (NOT deployed — models/vae_cc1.pt is unchanged)


## How to read this

- **Step 3/4** is the actual hyperparameter search — a joint grid, not the
  original sequential one — screened at a short budget, selected purely by
  validation loss (leak-free).
- **Step 6** is the honest comparison that matters: does the tuned
  configuration, fully trained, actually beat the deployed model's F1/PR-AUC
  on `cc1_test` and `drift_cc2`? If the winning grid point turns out to be
  the same as the original config (`latent=32, beta=0.01`), that is itself a
  legitimate, informative result — it would mean the original sequential
  search already found the right region of the space, not that tuning failed.
- This notebook does **not** overwrite `models/vae_cc1.pt` or
  `models/vae_cc1_eval.pkl` — the tuned model and its results are saved
  separately (`experiments/vae_cc1_tuned.pt`,
  `experiments/hyperparameter_tuning_results.pkl`) so the currently-deployed
  model stays untouched until a genuine improvement is confirmed and a
  decision is made to promote it.